# Running orbital-game on Apple Silicon (MPS)

How to install `jax-mps`, configure JAX before any `import jax`,
detect the MPS device at runtime, work within MLX's float32-only
constraint, and find the largest batch size that fits in memory.

Apple Silicon GPU support comes via `jax-mps`, which ships an MLX-
backed PJRT plugin. The plugin is experimental — some `jax`/`mctx`
ops are less optimized on MLX than on CUDA, and per-kernel dispatch
overhead is higher. The lever for speedup is batched work; see
`mcts_cpu_vs_mps.ipynb` for a `vmap`-over-states benchmark on this
framework.

## 1. Install

```
uv add 'orbital-game[mps]'        # or
pip install 'orbital-game[mps]'
```

`jax-mps` installs `jax_plugins/mps/` — JAX's plugin discovery picks
it up automatically the next time `import jax` runs.

## 2. Detect MPS at runtime

`JAX_PLATFORMS` controls which backends register on first `import jax`,
so it must be set *before* JAX is imported. When MPS is registered,
JAX picks it as the default device unless `JAX_DEFAULT_DEVICE` is also
set, which is rarely what you want for orbital-game (the package
enables `jax_enable_x64` for orbit precision, and MLX rejects
float64). Pin the default to CPU and opt into MPS per-call via
`jax.default_device(jax.devices('mps')[0])`.

The probe below uses `importlib.metadata.distribution` rather than
`importlib.util.find_spec('jax_plugins.mps')`, since the latter
raises `ModuleNotFoundError` on systems where the `jax_plugins`
namespace doesn't exist yet.

In [1]:
import importlib.metadata
import os

try:
    importlib.metadata.distribution('jax-mps')
    HAS_JAX_MPS = True
except importlib.metadata.PackageNotFoundError:
    HAS_JAX_MPS = False

if HAS_JAX_MPS:
    os.environ['JAX_PLATFORMS'] = 'cpu,mps'
    os.environ['JAX_DEFAULT_DEVICE'] = 'cpu'
else:
    os.environ.setdefault('JAX_PLATFORMS', 'cpu')

import jax
import jax.numpy as jnp

print('JAX', jax.__version__, '| default backend:', jax.default_backend())
print('CPU:', jax.devices('cpu'))
try:
    print('MPS:', jax.devices('mps'))
    HAS_MPS = True
except RuntimeError as e:
    print('MPS not registered:', e)
    HAS_MPS = False

Platform 'mps' is experimental and not all JAX functionality may be correctly supported!


JAX 0.10.0 | default backend: cpu
CPU: [CpuDevice(id=0)]
MPS: [MpsDevice(id=0)]


## 3. System info

`jax_plugins.mps.sysinfo.get_info()` reports the chip and total
system memory. Apple Silicon uses unified memory, so this is the
upper bound on what's available to MPS (less whatever the rest of
the system is using). Useful as a sanity check when picking batch
sizes.

In [2]:
if HAS_JAX_MPS:
    import jax_plugins.mps.sysinfo as sysinfo
    info = sysinfo.get_info()
    for k, v in info.items():
        print(f'{k:>10}: {v}')
else:
    print('jax-mps not installed — skipping sysinfo')

       jax: 0.10.0
    jaxlib: 0.10.0
   jax-mps: 0.10.0
    Python: 3.13.1
     macOS: 15.7.4 (24G517)
      Chip: Apple M1 Max
    Memory: 64 GB
   Backend: cpu


## 4. Float32-only constraint

MLX rejects float64. Two failure modes worth knowing:

1. `astrojax` enables `jax_enable_x64` at package import (orbit
   precision needs float64). `import orbital_game` does this for you.
2. With `JAX_PLATFORMS` including 'mps' and `JAX_DEFAULT_DEVICE`
   unset, plain `jnp.array([1.0, 2.0])` lands on MPS — and if x64
   is on, it errors immediately.

Three workarounds:

- pin `JAX_DEFAULT_DEVICE='cpu'` and only opt into MPS via
  `jax.default_device` (this notebook's pattern, also
  `lbg_ring_intercept.ipynb`),
- call `orbital_game.set_precision(jnp.float32)` to flip both
  `jax_enable_x64` and astrojax's dtype (and pay the orbit-precision
  cost on long horizons), or
- avoid `import orbital_game` until after MPS work is done.

Demonstrating the failure mode below; the runtime error from MLX is
the giveaway.

In [3]:
if HAS_MPS:
    jax.config.update('jax_enable_x64', True)
    try:
        with jax.default_device(jax.devices('mps')[0]):
            x = jnp.array([1.0, 2.0])  # float64 because x64 is on
            x.block_until_ready()
        print('unexpected success — array dtype was', x.dtype)
    except RuntimeError as e:
        first_line = str(e).splitlines()[0]
        print('expected failure:', first_line)
    jax.config.update('jax_enable_x64', False)

    # Same operation, float32: succeeds.
    with jax.default_device(jax.devices('mps')[0]):
        x = jnp.array([1.0, 2.0], dtype=jnp.float32)
        x.block_until_ready()
    print('float32 ok — dtype:', x.dtype, ', device:', x.devices())
else:
    print('MPS not available — skipping float32 demo')

expected failure: MLX does not support float64 (F64). Use jax.config.update('jax_enable_x64', False) or ensure your arrays are float32.
float32 ok — dtype: float32 , device: {MpsDevice(id=0)}


## 5. Running JIT'd code on MPS

Two equivalent patterns. `jax.default_device` makes any newly-
allocated array land on that device for the duration of the
context; `jax.device_put` moves an existing array.

For benchmarking or comparing devices, the context-manager pattern
is usually cleaner — wrap both array creation and the timed call
in the same `with` block.

In [4]:
target = jax.devices('mps')[0] if HAS_MPS else jax.devices('cpu')[0]

@jax.jit
def step(x):
    return jnp.tanh(x @ x.swapaxes(-1, -2)) @ x

key = jax.random.PRNGKey(0)

# Pattern A: default_device context.
with jax.default_device(target):
    x = jax.random.normal(key, (4, 32, 32), dtype=jnp.float32)
    y = step(x); y.block_until_ready()
print('A (default_device): result device =', y.devices())

# Pattern B: device_put an existing array.
x_cpu = jax.random.normal(key, (4, 32, 32), dtype=jnp.float32)
x_dev = jax.device_put(x_cpu, target)
y = step(x_dev); y.block_until_ready()
print('B (device_put):     result device =', y.devices())

A (default_device): result device = {MpsDevice(id=0)}
B (device_put):     result device = {MpsDevice(id=0)}


## 6. Tuning batch size

For a `vmap`-batched workload (MCTS, rollout sweep, ensemble forward
pass), the largest B that fits depends on per-lane memory plus what
else lives on the device. On Apple Silicon three things make naive
prediction unreliable:

1. **Process RSS doesn't capture Metal-private buffers.** MLX's GPU
   allocations don't show up in `Process.memory_info().rss`, so
   per-process memory probes miss most of the footprint.
2. **MLX's pool allocator is step-shaped.** It grabs system memory
   in big chunks — `psutil.virtual_memory().available` stays flat
   for several B values, then drops sharply when the pool expands.
   Linear extrapolation from small-B probes under-predicts OOM.
3. **Metal OOM can be uncatchable.** When the GPU's command buffer
   runs out, you sometimes get a Python `XlaRuntimeError`,
   sometimes a C++ `std::runtime_error` that terminates the kernel
   (`[METAL] kIOGPUCommandBufferCallbackErrorOutOfMemory`).

System-wide `psutil.virtual_memory().available` *does* capture MLX
allocations (Apple Silicon's unified memory means MLX comes out of
the same pool as everything else), at the cost of being coupled to
every other process running. The probe below walks B upward,
blocks until each call completes, checks system-wide `available`,
and stops before headroom drops below a safety floor. This is
conservative on purpose — it stops short of the cliff rather than
trying to drive over it and catch the fall.

If your machine is running other workloads (browser, build, tests)
that may grow during the probe, raise `safety_gb` to leave them
room.

In [5]:
import gc
import psutil


def find_max_batch(
    workload_factory,
    B_start: int = 1,
    B_max: int = 2**13,
    safety_gb: float = 8.0,
):
    """Probe increasing B; stop when system available memory drops below
    `safety_gb`, or when the workload raises a catchable OOM.

    Returns (last_safe_B, B_at_stop, reason). `reason` is one of
    'headroom', 'oom-<exc-type>', or None (sweep ran out at B_max).

    On Apple Silicon, MLX's GPU buffers contend with the rest of the
    system through unified memory, so `psutil.virtual_memory().available`
    is the authoritative signal — but it is system-wide and reflects
    every other process on the machine.

    Metal OOM can also fire as an uncatchable C++ exception that
    terminates the kernel. The headroom check stops *before* that line.
    Increase `safety_gb` if other workloads on the machine may grow
    during the probe.
    """
    last_ok = None
    B = B_start
    OOM_HINTS = ('memory', 'exhaust', 'oom', 'allocat', 'resource_exhausted')
    base_avail = psutil.virtual_memory().available
    print(f'  starting available: {base_avail/2**30:.2f} GB; safety floor: {safety_gb:.2f} GB')
    while B <= B_max:
        try:
            workload_factory(B)()
        except Exception as e:
            if not any(h in str(e).lower() for h in OOM_HINTS):
                raise
            print(f'  B={B:>6d}  caught {type(e).__name__}; stopping')
            return last_ok, B, f'oom-{type(e).__name__}'
        avail = psutil.virtual_memory().available
        used_gb = (base_avail - avail) / 2**30
        avail_gb = avail / 2**30
        if avail_gb < safety_gb:
            print(
                f'  B={B:>6d}  ok  (used so far ~{used_gb:+.2f} GB, '
                f'avail {avail_gb:.2f} GB < safety {safety_gb:.2f} GB; stopping)'
            )
            return B, B, 'headroom'
        print(f'  B={B:>6d}  ok  (used so far ~{used_gb:+.2f} GB, avail {avail_gb:.2f} GB)')
        last_ok = B
        B *= 2
        gc.collect()
    return last_ok, None, None


# Demo workload: vmapped tanh over (B, DIM, DIM) float32 arrays.
# Per-lane theoretical footprint = 4 * DIM**2 bytes (the input array).
# DIM=256 keeps the demo well within available memory on a 16 GB Mac
# while still triggering MLX's pool-expansion behavior at higher B.
DIM = 256
device = jax.devices('mps')[0] if HAS_MPS else jax.devices('cpu')[0]


@jax.jit
def reduce_tanh(x):
    return jnp.tanh(x).sum(axis=(-1, -2))


def workload_factory(B: int):
    def run():
        with jax.default_device(device):
            x = jax.random.normal(
                jax.random.PRNGKey(B), (B, DIM, DIM), dtype=jnp.float32
            )
            y = jax.vmap(reduce_tanh)(x)
            y.block_until_ready()
    return run


per_lane_mb = 4 * DIM * DIM / 2**20
print(f'Sweep: vmapped tanh over (B, {DIM}, {DIM}) float32 on {device}')
print(f'theoretical per-lane footprint: {per_lane_mb:.2f} MB (input array only)')
last_ok, B_stop, reason = find_max_batch(
    workload_factory, B_start=1, B_max=2**13, safety_gb=8.0
)
print()
print(f'largest safe B: {last_ok}    (stop reason: {reason})')

Sweep: vmapped tanh over (B, 256, 256) float32 on MPS:0
theoretical per-lane footprint: 0.25 MB (input array only)
  starting available: 26.06 GB; safety floor: 8.00 GB
  B=     1  ok  (used so far ~-0.00 GB, avail 26.07 GB)
  B=     2  ok  (used so far ~+0.01 GB, avail 26.05 GB)
  B=     4  ok  (used so far ~+0.03 GB, avail 26.03 GB)
  B=     8  ok  (used so far ~+0.09 GB, avail 25.98 GB)
  B=    16  ok  (used so far ~+0.21 GB, avail 25.85 GB)
  B=    32  ok  (used so far ~+0.65 GB, avail 25.42 GB)
  B=    64  ok  (used so far ~+1.19 GB, avail 24.87 GB)
  B=   128  ok  (used so far ~+1.83 GB, avail 24.23 GB)
  B=   256  ok  (used so far ~+3.31 GB, avail 22.76 GB)
  B=   512  ok  (used so far ~+5.98 GB, avail 20.09 GB)
  B=  1024  ok  (used so far ~+8.69 GB, avail 17.37 GB)
  B=  2048  ok  (used so far ~+16.22 GB, avail 9.85 GB)
  B=  4096  ok  (used so far ~+21.02 GB, avail 5.04 GB < safety 8.00 GB; stopping)

largest safe B: 4096    (stop reason: headroom)
